# 01 — Gradio Fundamentals

Build interactive ML web interfaces with Gradio.

**Concepts:** Blocks API, Components, Events, Layout, State

---

## Definition

Gradio is an open-source Python library for creating web-based UIs around ML models. With a few lines of Python, you get a browser interface with input widgets (text, image, audio, file) and output components — no HTML, CSS, or JavaScript required.

Two APIs:
- **`gr.Interface`**: single-function wrapper, one-shot demos
- **`gr.Blocks`**: full layout control with tabs, rows, columns, state

## Motivation

ML models need interfaces. Without Gradio you would:
- Write raw Flask/FastAPI endpoints + separate frontend code
- Handle file uploads, state management, input validation manually
- Deploy and maintain a full web application

Gradio collapses this to pure Python — ideal for rapid prototyping, demos, and portfolio projects.

## Theory: Components, Events, Layout

Every Gradio app is a **Component tree** inside a **Layout** wired by **Events**:

```
App (gr.Blocks)
├── Layout (Tabs / Rows / Columns)
│   ├── Input Component  (Textbox, File, Image, Audio, Number, Dropdown, Slider)
│   ├── Action Component (Button)
│   └── Output Component (Textbox, Image, Label, DataFrame, Plot)
└── Events (click, submit, change, upload)
    └── Callback function (Python) -> updates output(s)
```

**Event flow:** User interacts → event fires → Python callback runs → outputs update.

**State:** `gr.State` holds per-session data across interactions. Each browser tab gets its own state.

## Real-World Examples

| Use Case | Components |
|---|---|
| Chatbot | Chatbot + Textbox + State (history) |
| Image classifier | Image + Label + Button |
| Document Q&A | File + Textbox + State (document) |
| Model benchmark dashboard | DataFrame + Plot + Dropdown |

This project's `app.py` uses all these patterns across 5 tabs.

## Visual Explanation

**Block diagram of a Gradio app:**

```
┌─────────────────────────────────────────────────────┐
│  gr.Blocks(title="My App")                           │
│  ┌─────────────────────────────────────────────────┐│
│  │  gr.Markdown("# Title")                          ││
│  ├─────────────────────────────────────────────────┤│
│  │  gr.Row:                                         ││
│  │  ┌──────────────┐  ┌──────────────────────────┐ ││
│  │  │ gr.Textbox   │  │ gr.Button("Submit")       │ ││
│  │  │ (input)      │  │ (click → fn)             │ ││
│  │  └──────────────┘  └──────────────────────────┘ ││
│  ├─────────────────────────────────────────────────┤│
│  │  gr.Textbox(label="Output")                     ││
│  └─────────────────────────────────────────────────┘│
└─────────────────────────────────────────────────────┘
```

`gr.Interface(fn, inputs, outputs)` is syntactic sugar over this — creates layout automatically.

## Code Walkthrough

Run each cell below to see Gradio's APIs in action. Code blocks include inline comments explaining each pattern.

In [ ]:
import gradio as gr

# gr.Interface: quick single-function demo
# Wraps a Python function with auto-generated UI
def greet(name: str) -> str:
    return f"Hello {name}!"

demo = gr.Interface(fn=greet, inputs=gr.Textbox(label="Name"), outputs=gr.Textbox(label="Output"))
print("Interface created successfully")

In [ ]:
# gr.Blocks: full layout control
# with statement opens a block context — components are rendered in order
with gr.Blocks(title="Demo") as block:
    gr.Markdown("# My App")           # Static markdown cell
    inp = gr.Textbox(label="Input")   # Input component
    btn = gr.Button("Go")             # Action trigger
    out = gr.Textbox(label="Output")  # Output component
    btn.click(lambda x: x.upper(), inp, out)  # Wire event: click → fn → update

print("Blocks app ready")

In [ ]:
# Tabs: organize sections
# Each gr.Tab is a named container — only active tab renders
with gr.Blocks() as tabs_demo:
    with gr.Tab("Tab A"):
        gr.Markdown("Content A")
    with gr.Tab("Tab B"):
        gr.Markdown("Content B")

print("Tabs created")

In [ ]:
# gr.State: per-session persistent data
# counter holds the current count, unique per browser session
counter = gr.State(0)

def increment(count: int) -> tuple[int, int]:
    return count + 1, count + 1   # update state + display

with gr.Blocks() as state_demo:
    num = gr.Number(label="Count", value=0)
    btn = gr.Button("+")
    btn.click(increment, counter, [counter, num])  # multiple outputs

print("State demo ready")

In [ ]:
# File upload
# gr.File returns a tempfile-like object; None when no file
def handle_file(file) -> str:
    if file is None:
        return "No file"
    return f"Uploaded: {file.name} ({file.size} bytes)"

with gr.Blocks() as file_demo:
    f = gr.File(label="Upload")
    btn = gr.Button("Process")
    out = gr.Textbox()
    btn.click(handle_file, f, out)

print("File upload demo ready")

## Best Practices

1. **Use `gr.Blocks` for multi-tab apps** — `gr.Interface` doesn't support tabs
2. **Always set `label` on components** — improves accessibility and clarity
3. **Wire events with specific inputs** — `btn.click(fn, [inp1, inp2], [out1, out2])` for multiple I/O
4. **Use `gr.State` for session data** — never use global variables for per-user state
5. **Handle `None` in file/upload handlers** — users may not provide input
6. **Wrap everything in try/except** — errors crash the whole app without UI feedback
7. **Use `queue()` for long-running tasks** — prevents timeout on inference calls

## Common Mistakes

| Mistake | Fix |
|---|---|
| Forgetting `outputs` parameter | Always specify both `inputs` and `outputs` in `.click()` |
| Using `return` instead of `yield` for streaming | Streaming uses `yield`, single response uses `return` |
| Mutable default arguments in callbacks | Use `None` and initialize inside the function |
| Not closing files from upload handlers | File objects from `gr.File` are temporary — read immediately |
| Using `global` instead of `gr.State` | `gr.State` is per-session; globals are shared across all users |
| Binding events outside `gr.Blocks` context | All `.click()`, `.change()`, etc. must be inside `with gr.Blocks():` |

## Summary

- `gr.Interface` for quick single-function demos
- `gr.Blocks` for custom layouts with tabs, rows, columns
- `gr.State` preserves session state across interactions
- Components: Textbox, Button, Number, File, Image, Markdown
- Events: `.click()`, `.submit()`, `.change()`